# 자료 통합·공통 전처리

원본 `python/data_pooling.ipynb`, `tistory_pooling.ipynb`, `preprocess_1.ipynb`의 CSV 통합·제목/노이즈 필터를 공개용으로 정리했습니다. 300자(공백 포함) 이하 제외는 최종보고서에 적힌 기준을 후속 코드로 명시한 부분입니다. 원본의 수작업 검토·제외 내역이 없어 당시 최종 건수(5,044건·5,909건)의 완전 재현 코드는 아닙니다. 원문 CSV는 공개 저장소 밖에서만 처리합니다.

아래 형태소 열은 원본 `전체 코드.ipynb`의 `Okt` 형태소 분석 방식을 참고한 형태소 열 생성 예시입니다. 당시 `nva`와 `content_nouns`를 생성한 정확한 실행 코드는 확인되지 않았습니다. 이 출력으로 기존 저장 결과와 같은 키워드 순위·건수를 얻는다고 주장하지 않습니다. 일반 모델과 AI 모델의 구분도 이 코드에서 추정하지 않습니다.


In [ ]:
import os
import re
from pathlib import Path

import pandas as pd

# 비공개 원자료 폴더: 같은 컬럼으로 정리한 CSV를 입력
# title, content, date, link, platform
data_path = Path(os.environ['LG_VOC_PRIVATE_DATA'])
input_folder = data_path / 'input'
output_folder = data_path / 'processed'
output_folder.mkdir(parents=True, exist_ok=True)


In [ ]:
# 파일을 읽고 통합하는 함수
def load_and_combine(category, sources, folder_path):
    file_list = []
    desired_columns = ['title', 'content', 'date', 'link', 'platform']

    for source in sources:
        file_path = folder_path / f'{category}_{source}.csv'
        if not file_path.exists():
            print(f'File not found: {file_path}')
            continue
        df = pd.read_csv(file_path)
        df = df.rename(columns={'url': 'link', '제목': 'title',
                                '내용': 'content', '날짜': 'date'})
        if 'platform' not in df.columns:
            df['platform'] = source
        missing = [column for column in desired_columns if column not in df.columns]
        if missing:
            raise ValueError(f'{file_path.name}: 필요한 열이 없습니다. {missing}')
        file_list.append(df[desired_columns])

    if not file_list:
        raise ValueError(f'{category}: 통합할 파일이 없습니다.')
    combined_data = pd.concat(file_list, ignore_index=True)
    return combined_data


In [ ]:
# 불용어 목록: 원본 preprocess_1.ipynb의 제목 제외어
irrelevant_keywords = [
    '구매', '판매', '가격', '비교', '할인', '배송', '무료배송', '최저가', 'Top5',
    'Top 5', 'Top 10', 'Top10', '베스트5', '베스트10', '판매량', '꿀정보', '고민',
    '인기', '쇼핑', '증정', '순위', '가입', '핫딜', '예약', '세일', '만원',
    '출시', '팝니다', '삽니다', '알뜰', '무상', '어떤가요', '어떤거', '딜', '뉴스',
    '추천해주세요', '괜찮나요', '사드렸어요', '부탁드립니다', '있나요', '미개봉',
    '택포', '부탁드려요', '부탁', '해주세요', '쓰시는분', '인가요', '있나요',
    '있을까요', '괜찮은거', 'top3', '가요?', '당첨자', '혜택', '받아용', '쓰시나요',
    '받아요', '없나요', '좋나요', '살려는데', '시나요', '하나요', '쓰나요', '산다'
]

# 해당 단어가 title에 포함된 데이터를 제거
def remove_irrelevant_titles(data, keywords):
    # '원' 단독 검색은 '원룸'까지 제외하므로 가격 표현에만 적용
    pattern = '|'.join(re.escape(keyword) for keyword in keywords) + r'|\d+[\s,]*원'
    return data[~data['title'].str.contains(pattern, case=False, na=False)]


# 원본 노트북의 노이즈 문서 제외 조건
def should_drop(row):
    title = row['title']
    content = row['content']
    link_pattern = r'(http|https)://[^\s]+'
    if re.search(link_pattern, title) or re.search(link_pattern, content):
        return True
    hangul_pattern = r'^[ㄱ-ㅎㅏ-ㅣ\s]+$'
    if re.match(hangul_pattern, title) or re.match(hangul_pattern, content):
        return True
    english_only_pattern = r'^[a-zA-Z\s.,™-]+$'
    if re.match(english_only_pattern, title) or re.match(english_only_pattern, content):
        return True
    if title.lower().startswith('how to'):
        return True
    return False


def preprocess_data(data):
    data = data.copy()
    data['title'] = data['title'].fillna('').astype(str)
    data['content'] = data['content'].fillna('').astype(str)
    data['date'] = pd.to_datetime(data['date'], errors='coerce').dt.strftime('%Y-%m-%d')
    before = len(data)
    data = data[data['date'].between('2022-01-01', '2024-12-31')]
    print(f'게시일 미확인·기간 외 제외: {before - len(data)}건')

    before = len(data)
    data = data.drop_duplicates(subset='link').reset_index(drop=True)
    print(f'중복 링크 제외: {before - len(data)}건')
    before = len(data)
    data = remove_irrelevant_titles(data, irrelevant_keywords)
    print(f'제목 제외어 적용: {before - len(data)}건')
    before = len(data)
    data = data[~data.apply(should_drop, axis=1)]
    print(f'노이즈 문서 제외: {before - len(data)}건')

    # 최종보고서 기준을 후속 코드로 명시: 원본 실행 셀은 확인되지 않음
    before = len(data)
    data = data[data['content'].str.len() > 300]
    print(f'공백 포함 300자 이하 제외: {before - len(data)}건')
    return data.reset_index(drop=True)


In [ ]:
from konlpy.tag import Okt

# 원본 전체 코드.ipynb의 Okt(stem=True) 전처리 방식을 참고해 형태소 열 생성
stop_words = ['수', '있다', '을', '를', '이', '것', '같다', '으로', '와', '는', '은']

def tokenize_content(review, okt):
    review = re.sub(r'http[s]?://\S+|www\.\S+', '', review)
    review_text = re.sub(r'[^가-힣\s]', '', review)
    word_review = okt.pos(review_text, stem=True)

    nouns = [word for word, tag in word_review
             if tag == 'Noun' and word not in stop_words]
    verb_adj = [word for word, tag in word_review
                if tag in ['Verb', 'Adjective'] and word not in stop_words]
    nva = [word for word, tag in word_review
           if tag in ['Noun', 'Verb', 'Adjective'] and word not in stop_words]
    token = [word for word, tag in word_review if word not in stop_words]
    return (' '.join(nouns), ' '.join(verb_adj),
            ' '.join(nva), ' '.join(token))


def add_token_columns(data):
    okt = Okt()
    data = data.copy()
    tokens = [tokenize_content(content, okt) for content in data['content']]
    token_data = pd.DataFrame(tokens, columns=['noun', 'verb_adj', 'nva', 'token'], index=data.index)
    data = pd.concat([data, token_data], axis=1)
    data['content_nouns'] = data['noun']
    data['content_morphs'] = data['token']
    return data


In [ ]:
# 각 출처 CSV를 입력 폴더에 배치한 후 실행
sources = ['cafe1', 'cafe2', 'cafe3', 'blog', 'tistory', 'youtube']
analysis_folder = data_path / 'analysis_input'
analysis_folder.mkdir(parents=True, exist_ok=True)

for category in ['의류관리기', '무선스틱청소기']:
    pooled = load_and_combine(category, sources, input_folder)
    filtered = preprocess_data(pooled)
    output_path = output_folder / f'{category}_pool_filtered.csv'
    filtered.to_csv(output_path, index=False, encoding='utf-8-sig')

    # 공개 분석 노트북의 입력 열·파일명에 맞춰 저장
    # 두 파일은 같은 전처리 결과이며, 당시 원본의 서로 다른 버전은 아님
    tokenized = add_token_columns(filtered)
    for filename in [f'{category}.csv', f'{category}_수정.csv']:
        tokenized.to_csv(analysis_folder / filename, index=False, encoding='utf-8-sig')
    print(f'{category}: {len(pooled)}건 → {len(filtered)}건, {output_path.name}')
